# ONS Retail Sales Data Audit

This notebook examines the structure, coverage and quality of the Office for National Statistics Retail Sales Index time-series dataset.

**Source:** [ONS Retail Sales Index time series](https://www.ons.gov.uk/businessindustryandtrade/retailindustry/datasets/retailsales)  
**Release date:** 18 September 2026  
**Dataset ID:** DRSI

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [19]:
print("Working directory:", Path.cwd())
print("Data path:", data_path)
print("File exists:", data_path.exists())

Working directory: /Users/dafyddjones/projects/uk-retail-sales-analysis
Data path: data/raw/ons_retail_sales_index.csv
File exists: True


In [20]:
data_path = Path("data/raw/ons_retail_sales_index.csv")

retail_raw = pd.read_csv(data_path)

retail_raw.shape

(662, 622)

In [12]:
retail_raw["Title"].head(30).to_frame()

,Title
0,CDID
1,PreUnit
2,Unit
3,Release Date
4,Next release
5,Important Notes
6,1988
7,1989
8,1990
9,1991


In [13]:
retail_raw["Title"].tail(20).to_frame()

,Title
642,2025 JAN
643,2025 FEB
644,2025 MAR
645,2025 APR
646,2025 MAY
647,2025 JUN
648,2025 JUL
649,2025 AUG
650,2025 SEP
651,2025 OCT


In [18]:
retail_raw.iloc[:6, :6]

,Title,Implied deflator:AGG 1:Pred food stores - month on previous month%,Implied deflator:AGG 12:Pred non-food stores - month on previous month%,Implied deflator:47.19:Non-specialised stores - month on previous month%,"Implied deflator:AGG 5:Textile, clothing & footwear - month on previous month%",Implied deflator:AGG 7:Household goods stores - month on previous month%
0,CDID,A4RT,A4RU,A4RV,A4RW,A4RX
1,PreUnit,NaN,NaN,NaN,NaN,NaN
2,Unit,%,%,%,%,%
3,Release Date,18-09-2026,18-09-2026,18-09-2026,18-09-2026,18-09-2026
4,Next release,23 October 2026,23 October 2026,23 October 2026,23 October 2026,23 October 2026
5,Important Notes,NaN,NaN,NaN,NaN,NaN


In [21]:
metadata = retail_raw.iloc[:6, 1:].T

metadata.columns = [
    "cdid",
    "pre_unit",
    "unit",
    "release_date",
    "next_release",
    "important_notes",
]

metadata = metadata.rename_axis("series_title").reset_index()

metadata.head()

,series_title,cdid,pre_unit,unit,release_date,next_release,important_notes
0,Implied deflator:AGG 1:Pred food stores - mont...,A4RT,NaN,%,18-09-2026,23 October 2026,NaN
1,Implied deflator:AGG 12:Pred non-food stores -...,A4RU,NaN,%,18-09-2026,23 October 2026,NaN
2,Implied deflator:47.19:Non-specialised stores ...,A4RV,NaN,%,18-09-2026,23 October 2026,NaN
3,"Implied deflator:AGG 5:Textile, clothing & foo...",A4RW,NaN,%,18-09-2026,23 October 2026,NaN
4,Implied deflator:AGG 7:Household goods stores ...,A4RX,NaN,%,18-09-2026,23 October 2026,NaN


In [22]:
metadata.shape

(621, 7)

In [23]:
observations = retail_raw.iloc[6:].copy()

observations = observations.rename(columns={"Title": "period"})

observations.shape

(656, 622)

In [24]:
period_text = observations["period"].astype(str)

frequency_counts = pd.Series({
    "Annual": period_text.str.fullmatch(r"\d{4}").sum(),
    "Quarterly": period_text.str.fullmatch(r"\d{4} Q[1-4]").sum(),
    "Monthly": period_text.str.fullmatch(r"\d{4} [A-Z]{3}").sum(),
})

frequency_counts["Other"] = len(period_text) - frequency_counts.sum()

frequency_counts

Annual        38
Quarterly    154
Monthly      464
Other          0
dtype: int64

In [25]:
monthly = observations.loc[
    period_text.str.fullmatch(r"\d{4} [A-Z]{3}")
].copy()

monthly.insert(
    1,
    "date",
    pd.to_datetime(monthly["period"], format="%Y %b"),
)

monthly = monthly.sort_values("date").reset_index(drop=True)

monthly[["period", "date"]].iloc[[0, -1]]

,period,date
0,1988 JAN,1988-01-01
463,2026 AUG,2026-08-01


In [26]:
series_columns = retail_raw.columns[1:]

monthly[series_columns] = monthly[series_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

monthly[series_columns].dtypes.value_counts()

float64    621
Name: count, dtype: int64

In [27]:
coverage = pd.DataFrame({
    "non_missing": monthly[series_columns].notna().sum(),
    "missing": monthly[series_columns].isna().sum(),
})

coverage["coverage_pct"] = (
    coverage["non_missing"] / len(monthly) * 100
)

coverage.describe().round(1)

,non_missing,missing,coverage_pct
count,621.0,621.0,621.0
mean,421.4,42.6,90.8
std,79.0,79.0,17.0
min,200.0,0.0,43.1
25%,452.0,0.0,97.4
50%,452.0,12.0,97.4
75%,464.0,12.0,100.0
max,464.0,264.0,100.0


In [28]:
coverage_groups = pd.Series({
    "Complete series": coverage["missing"].eq(0).sum(),
    "Missing exactly 12 months": coverage["missing"].eq(12).sum(),
    "At least 90% coverage": coverage["coverage_pct"].ge(90).sum(),
    "Below 80% coverage": coverage["coverage_pct"].lt(80).sum(),
})

coverage_groups

Complete series              248
Missing exactly 12 months    205
At least 90% coverage        487
Below 80% coverage           125
dtype: int64

In [29]:
def count_internal_gaps(series):
    valid_positions = np.flatnonzero(series.notna())

    if len(valid_positions) == 0:
        return len(series)

    first_position = valid_positions[0]
    last_position = valid_positions[-1]

    return series.iloc[first_position:last_position + 1].isna().sum()


coverage["internal_gaps"] = monthly[series_columns].apply(
    count_internal_gaps
)

coverage["internal_gaps"].value_counts().sort_index().head(10)

internal_gaps
0    621
Name: count, dtype: int64

In [30]:
valid_values = monthly[series_columns].notna()

coverage["first_date"] = (
    valid_values.idxmax().map(monthly["date"])
)

coverage["last_date"] = (
    valid_values.iloc[::-1].idxmax().map(monthly["date"])
)

coverage.groupby(
    ["first_date", "last_date"]
).size().sort_values(ascending=False).head(10)

first_date  last_date 
1988-01-01  2026-08-01    248
1989-01-01  2026-08-01    205
2008-01-01  2026-08-01     40
2009-01-01  2026-08-01     21
1997-01-01  2026-08-01     18
1996-01-01  2026-08-01     16
1988-02-01  2026-08-01     12
1988-06-01  2026-08-01     10
1997-03-01  2026-08-01      8
1989-03-01  2026-08-01      8
dtype: int64

In [31]:
pd.set_option("display.max_colwidth", 120)

total_retail_candidates = metadata.loc[
    metadata["series_title"].str.contains(
        "All retail inc fuel",
        case=False,
        na=False,
    ),
    ["series_title", "cdid", "unit"],
]

total_retail_candidates

,series_title,cdid,unit
372,RSI:All retail inc fuel:All Business:VAL NSA:% change on same month a year ago,J59V,%
373,RSI:All retail inc fuel:Large Business:VAL NSA:% change on same month a year ago,J59W,%
374,RSI:All retail inc fuel:Small Business:VAL NSA:% change on same month a year ago,J59X,%
375,RSI:All retail inc fuel:All Business:VAL NSA:Rolling 3m on same 3m a year ago %,J5A8,%
379,RSI:Year to date % change:All retail inc fuel:All Business,J5AK,%
380,RSI:Year to date % change:All retail inc fuel:Large Business,J5AL,%
381,RSI:Year to date % change:All retail inc fuel:Small Business,J5AM,%
382,RSI:Year to date index:All retail inc fuel:All Business,J5AN,NaN
383,RSI:Year to date index:All retail inc fuel:Large Business,J5AO,NaN
384,RSI:Year to date index:All retail inc fuel:Small Business,J5AP,NaN


In [32]:
core_cdids = ["J5BS", "J5BT", "J5EB", "J5EC", "J5HW"]

core_metadata = metadata.loc[
    metadata["cdid"].isin(core_cdids),
    ["series_title", "cdid", "unit"],
].sort_values("cdid")

core_metadata

,series_title,cdid,unit
390,RSI:All retail inc fuel:All Business:VAL SA:% change on same month a year ago,J5BS,%
391,RSI:Month on prev month % change:All Business All retail inc fuel VAL SA,J5BT,%
414,RSI:All retail inc fuel:All Business:VOL SA:% change on same month a year ago,J5EB,%
415,RSI:Month on prev month % change:All Business All retail inc fuel VOL SA,J5EC,%
422,RSI:Implied deflator:AGG21:All retail inc fuel - year on year%,J5HW,%


In [33]:
cdid_to_title = metadata.set_index("cdid")["series_title"]

selected_series = {
    "J5BS": "value_yoy_pct",
    "J5BT": "value_mom_pct",
    "J5EB": "volume_yoy_pct",
    "J5EC": "volume_mom_pct",
    "J5HW": "implied_deflator_yoy_pct",
}

core_data = monthly[
    ["date"] + [
        cdid_to_title[cdid]
        for cdid in selected_series
    ]
].copy()

core_data = core_data.rename(
    columns={
        cdid_to_title[cdid]: new_name
        for cdid, new_name in selected_series.items()
    }
)

core_data.tail()

,date,value_yoy_pct,value_mom_pct,volume_yoy_pct,volume_mom_pct,implied_deflator_yoy_pct
459,2026-04-01,4.6,0.6,0.0,-0.9,4.7
460,2026-05-01,7.2,0.8,3.2,1.2,3.9
461,2026-06-01,6.7,0.4,3.4,0.6,3.2
462,2026-07-01,4.0,-0.6,1.2,-0.5,2.8
463,2026-08-01,6.0,1.3,2.4,0.5,3.5


In [34]:
core_data.isna().sum()

date                          0
value_yoy_pct               108
value_mom_pct                97
volume_yoy_pct              108
volume_mom_pct               97
implied_deflator_yoy_pct    108
dtype: int64

In [35]:
core_data.set_index("date").apply(
    lambda series: series.first_valid_index()
)

value_yoy_pct              1997-01-01
value_mom_pct              1996-02-01
volume_yoy_pct             1997-01-01
volume_mom_pct             1996-02-01
implied_deflator_yoy_pct   1997-01-01
dtype: datetime64[us]

In [36]:
core_complete = core_data.dropna().reset_index(drop=True)

print("Shape:", core_complete.shape)

core_complete[["date"]].agg(["min", "max"])

Shape: (356, 6)


,date
min,1997-01-01
max,2026-08-01


In [37]:
expected_dates = pd.date_range(
    start=core_complete["date"].min(),
    end=core_complete["date"].max(),
    freq="MS",
)

print("Expected months:", len(expected_dates))
print("Actual rows:", len(core_complete))
print(
    "Continuous monthly series:",
    core_complete["date"].equals(pd.Series(expected_dates)),
)

Expected months: 356
Actual rows: 356
Continuous monthly series: True


In [38]:
print(
    "Duplicate dates:",
    core_complete["date"].duplicated().sum(),
)

core_complete.drop(columns="date").describe().T.round(2)

Duplicate dates: 0


,count,mean,std,min,25%,50%,75%,max
value_yoy_pct,356.0,3.46,3.81,-24.0,2.20,3.5,4.62,43.5
value_mom_pct,356.0,0.31,1.91,-18.9,-0.30,0.3,0.90,13.2
volume_yoy_pct,356.0,1.90,4.20,-23.0,0.20,1.9,3.70,41.0
volume_mom_pct,356.0,0.18,1.86,-17.8,-0.43,0.2,0.72,13.2
implied_deflator_yoy_pct,356.0,1.60,3.04,-5.1,0.10,1.2,2.80,13.0


In [39]:
extreme_rows = []

for column in core_complete.columns.drop("date"):
    minimum_index = core_complete[column].idxmin()
    maximum_index = core_complete[column].idxmax()

    extreme_rows.append({
        "indicator": column,
        "minimum": core_complete.loc[minimum_index, column],
        "minimum_date": core_complete.loc[minimum_index, "date"],
        "maximum": core_complete.loc[maximum_index, column],
        "maximum_date": core_complete.loc[maximum_index, "date"],
    })

extremes = pd.DataFrame(extreme_rows)

extremes

,indicator,minimum,minimum_date,maximum,maximum_date
0,value_yoy_pct,-24.0,2020-04-01,43.5,2021-04-01
1,value_mom_pct,-18.9,2020-04-01,13.2,2020-05-01
2,volume_yoy_pct,-23.0,2020-04-01,41.0,2021-04-01
3,volume_mom_pct,-17.8,2020-04-01,13.2,2020-05-01
4,implied_deflator_yoy_pct,-5.1,2001-11-01,13.0,2022-07-01


In [40]:
latest_indicators = (
    core_complete.tail(1)
    .set_index("date")
    .T
    .rename(columns={core_complete["date"].iloc[-1]: "latest_value"})
)

latest_indicators

date,latest_value
value_yoy_pct,6.0
value_mom_pct,1.3
volume_yoy_pct,2.4
volume_mom_pct,0.5
implied_deflator_yoy_pct,3.5


In [41]:
processed_path = Path(
    "data/processed/retail_sales_core_monthly.csv"
)

core_complete.to_csv(
    processed_path,
    index=False,
    date_format="%Y-%m-%d",
)

print("Saved to:", processed_path)
print("Rows exported:", len(core_complete))

Saved to: data/processed/retail_sales_core_monthly.csv
Rows exported: 356


## Audit conclusion

- The raw ONS dataset contains 662 rows and 622 columns.
- It includes metadata for 621 retail time series.
- The observation data contains 38 annual, 154 quarterly and 464 monthly periods.
- Monthly coverage runs from January 1988 to August 2026.
- Missing values occur before or after individual series' available periods; there are no internal gaps.
- Five core indicators were selected for total retail sales including fuel:
  - year-on-year value growth;
  - month-on-month value growth;
  - year-on-year volume growth;
  - month-on-month volume growth; and
  - year-on-year implied price growth.
- The complete core dataset contains 356 continuous monthly observations from January 1997 to August 2026.
- There are no duplicate dates or missing values in the core dataset.
- Extreme observations align with identifiable economic events, particularly COVID-19 and the 2022 inflation shock.
- The cleaned dataset was exported to `data/processed/retail_sales_core_monthly.csv`.

The next notebook will perform exploratory trend analysis and visualisation.